# Agentic AI, Day 2 Lab: Routing, Parallelization, Reflection

Yesterday you built an agent: a model in a loop that can call your tools. Today you make it smarter in three ways.

1. **Routing:** classify each request and send it down the right branch.
2. **Parallelization:** run independent subtasks at the same time, then merge.
3. **Reflection:** have the model critique its own answer and improve it.

Everything still runs locally on Ollama, no cloud keys. We start by reloading the Day 1 pieces so this notebook runs on its own, then build the three patterns one at a time, and finish by combining them.

Run the cells in order, top to bottom.

## Setup and a quick reload of Day 1

Same setup as yesterday: Ollama running, a tool-capable model pulled (`llama3.1:8b` or `qwen2.5:7b`), and the `ollama` Python client installed.

The cell below reloads the tools and the agent loop you wrote on Day 1: `ask`, `get_weather`, `calculate`, and `run_agent`. Read it to refresh your memory, then run it. We build on these all afternoon.

**If your machine does not have `llama3.1`, change `MODEL` to a model you have pulled.**

In [1]:
!ollama list

]11;?\NAME              ID              SIZE      MODIFIED     
llama3.1:8b       46e0c10c039e    4.9 GB    24 hours ago    
qwen3.6:latest    07d35212591f    23 GB     27 hours ago    
gemma4:latest     c6eb396dbd59    9.6 GB    8 days ago      
qwen3.5:latest    6488c96fa5fa    6.6 GB    4 weeks ago     
llama3:latest     365c0bd3c000    4.7 GB    4 weeks ago     


In [2]:
import ollama, ast, operator

MODEL = "gemma4:latest"   # change to a tool-capable model you have pulled

def ask(prompt: str) -> str:
    """Send one prompt to the model and return the reply text."""
    r = ollama.chat(model=MODEL, messages=[{"role": "user", "content": prompt}])
    return r.message.content

# ---- tools from Day 1 ----
def get_weather(city: str) -> str:
    """Get the current weather for a city.

    Args:
        city: The name of the city, for example "Pune".
    """
    data = {"pune": "31C, clear sky", "mumbai": "33C, humid",
            "delhi": "29C, hazy", "bangalore": "26C, light rain"}
    return data.get(city.lower(), f"No weather data for {city}")

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}
def _ev(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_ev(node.left), _ev(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_ev(node.operand))
    raise ValueError("Unsupported expression")
def calculate(expression: str) -> str:
    """Evaluate a basic arithmetic expression. Supports + - * / ** and parentheses.

    Args:
        expression: A math expression, for example "31 * 9 / 5 + 32".
    """
    return str(_ev(ast.parse(expression, mode="eval").body))

# ---- the robust agent loop from Day 1 ----
def run_agent(question, tools_list, tools_map, max_steps=6):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_steps):
        res = ollama.chat(model=MODEL, messages=messages, tools=tools_list)
        messages.append(res.message)
        if not res.message.tool_calls:
            return res.message.content
        for call in res.message.tool_calls:
            name = call.function.name
            args = call.function.arguments or {}
            if name not in tools_map:
                result = f"Error: no tool named '{name}'."
            else:
                try:
                    result = str(tools_map[name](**args))
                except Exception as e:
                    result = f"Error running {name}: {e}"
            messages.append({"role": "tool", "content": result})
    return "Stopped: reached the step limit."

print("Day 1 tools and agent loaded. Using MODEL =", MODEL)

Day 1 tools and agent loaded. Using MODEL = gemma4:latest


## Milestone 1: Build a router

A router does one job: read a request and decide what kind it is. We will classify each question into one of three labels: `weather`, `math`, or `general`.

Two important design choices, straight from the lecture:
- **Constrain the output.** We ask the model to reply with a single word, which makes the result trivial to parse.
- **Always fall back.** If the model returns something unexpected, we default to `general` rather than crashing.

In [3]:
def route(question: str) -> str:
    """Classify a question into exactly one label: weather, math, or general."""
    prompt = (
        "Classify the user question into exactly one word: "
        "weather, math, or general. Reply with only that one word.\n\n"
        f"Question: {question}"
    )
    label = ask(prompt).strip().lower()
    # keep only a known label, otherwise fall back to general
    for known in ("weather", "math", "general"):
        if known in label:
            return known
    return "general"

# Test the router on a few questions. Watch the label it picks.
for q in ["What is the weather in Pune?",
          "What is 18 times 7?",
          "Tell me a fun fact about octopuses."]:
    print(f"{route(q):8}  <-  {q}")

weather   <-  What is the weather in Pune?
math      <-  What is 18 times 7?
general   <-  Tell me a fun fact about octopuses.


**What you should see:** `weather`, `math`, `general`, one per line.

**Your turn:** add a question of your own to the list and run it again. Try an ambiguous one, like "how warm is 30 degrees times two", and see which label wins. Ambiguous inputs are exactly where routing gets hard.

## Milestone 2: Wire the branches

Now we connect each label to a handler. The weather and math branches reuse the Day 1 agent with just the relevant tool. The general branch is a plain model call.

Notice the `print` inside `handle`: logging the chosen label is the single most useful habit for catching misroutes during development. A silent misroute is very hard to debug.

In [4]:
def weather_branch(q): return run_agent(q, [get_weather], {"get_weather": get_weather})
def math_branch(q):    return run_agent(q, [calculate], {"calculate": calculate})
def general_branch(q): return ask(q)

def handle(question: str) -> str:
    label = route(question)
    print(f"[router] -> {label}")     # log the decision so misroutes are visible
    if label == "weather":
        return weather_branch(question)
    elif label == "math":
        return math_branch(question)
    else:
        return general_branch(question)

print(handle("What is the weather in Mumbai?"))
print()
print(handle("What is 144 divided by 12?"))
print()
print(handle("Who wrote the play Hamlet?"))

[router] -> weather
The weather in Mumbai is 33°C and humid.

[router] -> math
The answer is 12.

[router] -> general
William Shakespeare wrote the play *Hamlet*.


**What you should see:** each question prints its router label, then the right branch answers. Weather and math route to the tool-using agent, everything else to the plain branch.

You have now built explicit routing: your classifier, your branches, full visibility into the decision. Compare this to Day 1, where the model picked the tool for you. Both are routing. This version gives you more control.

## Milestone 3: Parallel batch

Some work is just the same task repeated over many items, and those items do not depend on each other. Running them one after another wastes time. We run them together with a thread pool, then merge the results.

We will summarise a list of product reviews, first sequentially, then in parallel, and time both so you can see the difference.

In [5]:
import time
from concurrent.futures import ThreadPoolExecutor

reviews = [
    "Battery lasts all day and the screen is gorgeous, but it is a bit pricey.",
    "Stopped working after a week and the support was terrible.",
    "Great value, fast delivery, I would buy it again.",
    "Average phone, nothing special, the camera is weak.",
    "Loved it at first but it overheats badly during games.",
]

def summarise(review: str) -> str:
    return ask(f"Summarise this review in 5 words or fewer: {review}")

# sequential: one after another
start = time.perf_counter()
seq = [summarise(r) for r in reviews]
seq_time = time.perf_counter() - start

# parallel: all at once with a thread pool
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    par = list(pool.map(summarise, reviews))
par_time = time.perf_counter() - start

# merge step: here we just collect them into one block of text
for s in par:
    print("-", s)
print(f"\nsequential: {seq_time:.1f}s     parallel: {par_time:.1f}s")

- Excellent, pricey, long-lasting.
- Failed fast, terrible support.
- Great value, fast delivery.
- Mediocre phone, poor camera.
- Overheats badly during gameplay.

sequential: 4.2s     parallel: 4.1s


**What you should see:** five short summaries, then a timing line. The parallel run should be faster, though how much faster depends on your machine.

**An honest note about local models.** A single Ollama server processes a limited number of requests at once, controlled by the `OLLAMA_NUM_PARALLEL` setting. If it is set to 1, your parallel calls quietly queue and you see little speedup. The pattern is still correct, and on real infrastructure (or with a higher parallel setting) the gain is large. The lesson is the shape of the code, `pool.map` plus a merge, not the exact number of seconds.

**The merge step** here is simple: we print the list. Sometimes you want a smarter merge, for example one more model call that combines all summaries into a single verdict. Try that as a challenge at the end.

## Milestone 4: Reflection loop

Reflection adds a self-improvement step: generate a draft, critique it, then revise using the critique. It is just prompt chaining with a specific shape.

We will ask the model to write a small function, then improve it. Two design points:
- We **cap the number of rounds** so it cannot loop forever.
- We let the critic say `LGTM` (looks good to me) to **stop early** when the draft is already fine.

In [6]:
def reflect(task: str, rounds: int = 1) -> str:
    """Generate a function for the task, then critique and revise it."""
    draft = ask(f"Write a short Python function that {task}. Return only the code.")

    for i in range(rounds):
        critique = ask(
            "Review this Python code for correctness, edge cases, and clarity. "
            "List concrete problems. If it is already good, reply with just 'LGTM'.\n\n"
            f"{draft}"
        )
        print(f"[round {i}] critique: {critique[:80]}...")
        if "lgtm" in critique.lower():
            break
        draft = ask(
            "Rewrite the code to fix these issues. Return only the code.\n\n"
            f"Issues:\n{critique}\n\nCurrent code:\n{draft}"
        )
    return draft

print(reflect("returns the n-th Fibonacci number", rounds=2))

[round 0] critique: LGTM...
```python
def fibonacci(n):
    if n < 0:
        raise ValueError("Input must be a non-negative integer")
    if n <= 1:
        return n
    
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b
```


**What you should see:** one or two critique lines, then a final function that is usually cleaner than a first pass (handles edge cases, clearer names).

**Your turn:** change the task to something with obvious edge cases, like "checks whether a string is a palindrome, ignoring spaces and case", and watch the critique catch them.

**The strongest critic is often not the model.** For code, the best critique is to actually run it and check the result. Asking a small model whether its own code is correct is weaker than running a test. We do the prompt-based version here because it is simple, but keep that in mind.

## Milestone 5: Combine routing and reflection

Now stack two patterns. We route the question to the right branch for a first answer, then reflect once on that answer to improve it before returning. Routing picks the path, reflection raises the quality.

In [7]:
def smart_answer(question: str) -> str:
    """Route to the right branch, then reflect once on the answer."""
    draft = handle(question)          # routing happens inside handle()

    critique = ask(
        f"Review this answer to the question '{question}'. "
        "Point out anything wrong, unclear, or missing. "
        f"If it is already good, reply with just 'LGTM'.\n\n{draft}"
    )
    if "lgtm" in critique.lower():
        return draft

    improved = ask(
        "Improve the answer using this critique. Keep it concise.\n\n"
        f"Critique:\n{critique}\n\nAnswer:\n{draft}"
    )
    return improved

print(smart_answer("Explain in two sentences why the sky is blue."))

[router] -> general
Sunlight is composed of all visible colors, and when it reaches Earth, it encounters molecules and particles in our atmosphere. These tiny atmospheric components scatter shorter wavelengths of light—specifically blue—more effectively than longer wavelengths, redirecting the blue light across the entire sky.


**What you should see:** the router label, then a first answer, then (usually) an improved final answer. The patterns compose: each one stays simple on its own, and together they make a noticeably better system.

## Milestone 6: Measure

The mature habit is to check whether the fancy version is actually better, not just assume it. Reflection costs extra calls and time, so let us compare a plain answer against a reflected one, on both quality and speed.

In [8]:
import time

def timed(fn, arg):
    start = time.perf_counter()
    out = fn(arg)
    return out, time.perf_counter() - start

q = "Give three tips for writing clean Python functions."

plain, t_plain = timed(general_branch, q)
smart, t_smart = timed(smart_answer, q)

print(f"PLAIN  ({t_plain:.1f}s):\n{plain}\n")
print(f"SMART  ({t_smart:.1f}s):\n{smart}")

[router] -> general
PLAIN  (107.9s):
Writing clean functions is primarily about **readability** and **maintainability**. If you (or someone else) can understand what the function does just by looking at its signature and body, it's probably clean.

Here are three core tips:

***

### 1. Adhere to the Single Responsibility Principle (SRP)

**What it means:** A function should do one thing, and do it well. If you find yourself using helper comments like `# Handles file loading AND validation AND network transmission`, the function is doing too much.

**The goal:** Smaller functions are easier to test, easier to debug, and less prone to unintended side effects.

**How to apply it:**
*   **Decompose:** If a function has more than 15–20 lines of code, ask yourself, "What are the distinct steps here?"
*   **Break it out:** Turn each distinct step into its own function. For example, instead of one function that `loads_data_and_processes_it_and_saves_results()`, create three smaller, focused f

**What you should see:** two answers and two timings. The smart version takes longer (more model calls) and is usually more complete or better organised. Now you can make an informed choice: is the quality gain worth the extra latency for this use case? That judgement, not the code, is the real skill.

## You did it

Today you added three patterns on top of yesterday's agent:

- **Routing:** a classifier that sends each request to the right branch, with a fallback.
- **Parallelization:** the same task run over many items at once with a thread pool, then merged.
- **Reflection:** generate, critique, revise, with a round cap and an early stop.
- And you **combined and measured** them, which is how you decide what is actually worth using.

### Optional challenges

1. **Smarter merge:** in Milestone 3, add a final model call that combines the five review summaries into a single overall verdict.
2. **Rules-based router:** write a version of `route` that uses keywords (no model call) and compare its speed and accuracy to the LLM router.
3. **Tools as critic:** in `reflect`, instead of asking the model to judge the code, actually run it on a test input and feed the real result back as the critique.
4. **Two-round combine:** let `smart_answer` reflect up to two times, stopping early on `LGTM`.

### What comes next

Tomorrow, Day 3, the model starts planning its own steps: **Planning** and the **ReAct** loop, where the model reasons, acts, observes, and decides the next step entirely on its own. Today you wired the control flow. Tomorrow the model wires it for you.